In [1]:
from dnallm import load_config, load_model_and_tokenizer, DNADataset, DNATrainer
from dnallm.datahandling import show_preset_dataset, load_preset_dataset

In [2]:
# Load the config file
configs = load_config("./finetune_config.yaml")
configs["task"].task_type = "regression"
configs["task"].num_labels = 1
configs["task"].label_names = ["promoter strength"]
configs["finetune"].per_device_train_batch_size = 24
configs["finetune"].per_device_eval_batch_size = 24
configs["finetune"].num_train_epochs = 2
configs["finetune"].logging_steps = 400
configs["finetune"].eval_steps = 400
configs["finetune"].save_steps = 400
configs["finetune"].learning_rate = 2e-5
configs["finetune"].metric_for_best_model = "r2"
configs["finetune"].output_dir = "outputs_regression_plantnt_singlebase"
configs["finetune"].save_safetensors = False

In [3]:
# Load the model and tokenizer
model_name = "zhangtaolab/plant-nucleotide-transformer-singlebase"
# from ModelScope
model, tokenizer = load_model_and_tokenizer(model_name, task_config=configs["task"], source="modelscope")

2026-04-04 17:33:01,087 - modelscope - INFO - Not logged-in, you can login for uploadingor accessing controlled entities.


17:33:01 - dnallm.utils.support - INFO - Model files are stored in /home/liuguanqing/.cache/modelscope/hub/models/lgq12697/plant-nucleotide-transformer-singlebase


Some weights of the model checkpoint at /home/liuguanqing/.cache/modelscope/hub/models/lgq12697/plant-nucleotide-transformer-singlebase were not used when initializing EsmForSequenceClassification: ['lm_head.bias', 'lm_head.decoder.weight', 'lm_head.dense.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing EsmForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing EsmForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of EsmForSequenceClassification were not initialized from the model checkpoint at /home/liuguanqing/.cache/modelscope/hub/models/lgq12697/plant-nucleotid

In [4]:
dataset_name = "plant-genomic-benchmark"
PLANT_DATASET = show_preset_dataset()[dataset_name]
print(PLANT_DATASET["tasks"])

['poly_a.arabidopsis_thaliana', 'poly_a.oryza_sativa_indica', 'poly_a.oryza_sativa_japonica', 'poly_a.chlamydomonas_reinhardtii', 'poly_a.medicago_truncatula', 'poly_a.trifolium_pratense', 'splicing.arabidopsis_thaliana_acceptor', 'splicing.arabidopsis_thaliana_donor', 'lncrna.m_esculenta', 'lncrna.z_mays', 'lncrna.g_max', 'lncrna.s_lycopersicum', 'lncrna.t_aestivum', 'lncrna.s_bicolor', 'promoter_strength.leaf', 'promoter_strength.protoplast', 'terminator_strength.leaf', 'terminator_strength.protoplast', 'gene_exp.arabidopsis_thaliana', 'gene_exp.oryza_sativa', 'gene_exp.zea_mays', 'gene_exp.glycine_max', 'gene_exp.solanum_lycopersicum', 'pro_seq.m_esculenta', 'chromatin_access.arabidopis_thaliana', 'chromatin_access.oryza_sativa_MH63', 'chromatin_access.oryza_sativa_ZS97', 'chromatin_access.brachypodium_distachyon', 'chromatin_access.zea_mays', 'chromatin_access.sorghum_bicolor', 'chromatin_access.setaria_italica']


In [5]:
# Load a preset dataset
# datasets = DNADataset.load_local_data(dataset_path, seq_col="sequence", label_col="label", tokenizer=tokenizer)
task = "promoter_strength.protoplast"
datasets = load_preset_dataset(dataset_name, task=task)

# Encode the datasets
datasets.encode_sequences(tokenizer=tokenizer)

Loading dataset: lgq12697/plant-genomic-benchmark ...
promoter_strength.protoplast


Encoding inputs:   0%|          | 0/61051 [00:00<?, ? examples/s]

Encoding inputs:   0%|          | 0/7595 [00:00<?, ? examples/s]

In [6]:
# Initialize the trainer
trainer = DNATrainer(
    model=model,
    config=configs,
    datasets=datasets
)

In [7]:
# Start training
metrics = trainer.train()
print(metrics)

Step,Training Loss,Validation Loss,Mse,Mae,R2,Spearmanr
400,1.126700,0.701745,0.701745,0.649551,0.543000,0.726835
800,0.631800,0.597692,0.597692,0.601097,0.610000,0.779515
1200,0.554000,0.579766,0.579766,0.596009,0.622000,0.795076
1600,0.519800,0.488322,0.488322,0.536077,0.682000,0.804085
2000,0.494200,0.505233,0.505233,0.549724,0.671000,0.799873
2400,0.483200,0.487148,0.487148,0.539901,0.682000,0.809226
2800,0.441100,0.466668,0.466668,0.526544,0.696000,0.818098
3200,0.408500,0.446801,0.446801,0.511622,0.709000,0.820219
3600,0.399400,0.451471,0.451471,0.513134,0.706000,0.823075
4000,0.394700,0.439216,0.439216,0.506951,0.714000,0.823544


{'train_runtime': 11062.8979, 'train_samples_per_second': 11.037, 'train_steps_per_second': 0.46, 'total_flos': 6.949004332211405e+16, 'train_loss': 0.5090990216477113, 'epoch': 2.0}


In [8]:
# Do prediction on the test set
predictions = trainer.infer()
predictions.metrics

{'test_loss': 0.4371240735054016,
 'test_mse': 0.4371240653281628,
 'test_mae': 0.5049905701714315,
 'test_r2': 0.715,
 'test_spearmanr': 0.8263456169784202,
 'test_runtime': 185.3546,
 'test_samples_per_second': 40.976,
 'test_steps_per_second': 1.71}

In [9]:
from dnallm import DNAInference
from dnallm.configuration import InferenceConfig
from dnallm.inference.plot import plot_scatter

In [10]:
configs["inference"] = InferenceConfig(
    batch_size=200,
    max_length=512,
    device="auto"
)

In [13]:
!cp {model.model_dir}/*.py {configs["finetune"].output_dir}
# !sed -i 's/from .flash_attn_triton import flash_attn_qkvpacked_func/flash_attn_qkvpacked_func = None/g' {configs["finetune"].output_dir}/bert_layers.py

In [14]:
model_path = configs["finetune"].output_dir
model, tokenizer = load_model_and_tokenizer(model_path, task_config=configs["task"], source="local")

In [15]:
# Create inference engine
inference_engine = DNAInference(
    model=model,
    tokenizer=tokenizer,
    config=configs
)

20:57:17 - dnallm.utils.support - INFO - Using device: cuda


In [16]:
data = datasets.dataset["test"]
data = data.remove_columns([x for x in data.column_names if x not in ["labels", "input_ids", "attention_mask"]])
results, metrics = inference_engine.infer_file(data, evaluate=True, plot_metrics=True)
print(metrics)

Inferring: 100%|██████████| 38/38 [02:57<00:00,  4.68s/it]


{'mse': 0.4371240642200206, 'mae': 0.5049905701557358, 'r2': 0.715, 'spearmanr': 0.8263456093034578, 'scatter': {'predicted': array([ 0.7578414 ,  0.84711194, -0.3517361 , ..., -0.8397565 ,
       -1.478601  , -1.4771808 ], dtype=float32), 'experiment': tensor([ 0.3665, -0.1015, -0.4879,  ..., -0.2059, -0.6267, -1.2771])}}


In [17]:
metrics["scatter"]["r2"] = metrics["r2"]
plot_scatter({"Plant NT singlebase": metrics["scatter"]}, show_score=True)

alt.LayerChart(...)